# PPO VM Allocation Experiments

This notebook trains and evaluates PPO agents for VM allocation using the existing DRL pipeline:

- `rl/environment.py`: `VMAllocationEnv` (Gymnasium environment)
- `rl/config.py`: PPO and reward configuration
- `train_ppo.py`: training script
- `eval_ppo.py`: evaluation script

You can use this notebook to:
- Run quick training experiments (e.g., smaller `total_timesteps`)
- Evaluate PPO and inspect per-step schedules and metrics

**Note**: LP vs PPO comparison is done in `lp_vs_ppo_comparison.ipynb`.


In [5]:
# Imports and configuration

from pathlib import Path

# Module imports
import importlib
import train_ppo as train_ppo_module
import eval_ppo as eval_ppo_module
import rl.config as rl_config_module

from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST
from train_ppo import train_ppo
from eval_ppo import evaluate_scenario, print_comparison

# Reload modules to pick up latest code when notebook stays open
importlib.reload(train_ppo_module)
importlib.reload(eval_ppo_module)
importlib.reload(rl_config_module)

# Refresh config after reload
from rl.config import PPOConfig, SCENARIO_OVERLOAD, SCENARIO_COST

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)

# Show current default PPO configuration
config = PPOConfig()
config


Project root: e:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure


PPOConfig(learning_rate=0.0003, n_steps=2048, batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, clip_range_vf=None, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5, episode_length=480, horizon=120, total_timesteps=1000000, tensorboard_log='./tensorboard_logs/', log_interval=10, save_freq=10000)

In [6]:
# Configuration (for training or evaluation)
from copy import deepcopy

# Number of parallel envs (increase if you have CPU/GPU resources)
N_ENVS = 8

exp_config = deepcopy(config)

# Training settings (if you want to train)
exp_config.total_timesteps = 1_000_000

# Episode length for evaluation:
# - 480 = 4 hours (default, same as training)
# - 17150 = full test set (~6 days)
exp_config.episode_length = 17150  # evaluate full test set

print("Configuration:")
print("  total_timesteps =", exp_config.total_timesteps, "(for training)")
print("  episode_length  =", exp_config.episode_length, "steps (for evaluation)")
print("  n_envs          =", N_ENVS)


Configuration:
  total_timesteps = 1000000 (for training)
  episode_length  = 17150 steps (for evaluation)
  n_envs          = 8


In [ ]:

print("Training configuration:")
print("  total_timesteps =", exp_config.total_timesteps)
print("  episode_length  =", exp_config.episode_length)
print("  horizon         =", exp_config.horizon)
print("  n_envs          =", N_ENVS)

# Train overload-first scenario
model_overload = train_ppo(
    scenario=SCENARIO_OVERLOAD,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)

# Train cost-first scenario
model_cost = train_ppo(
    scenario=SCENARIO_COST,
    config=exp_config,
    total_timesteps=exp_config.total_timesteps,
    n_envs=N_ENVS,
    continue_training=False,
)


Training configuration:
  total_timesteps = 1000000
  episode_length  = 17150
  horizon         = 120
  n_envs          = 8

Training PPO for scenario: OVERLOAD
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_forest
  Target: cpu_total_usage
  Saved at: 2025-11-11 22:19:30
Latest model: models\svr_memory_usage_pct_20251111_184735.pkl
✓ Model loaded: models\svr_memory_usage_pct_20251111_184735.pkl
  Model: svr
  Target: memory_usage_pct
  Saved at: 2025-11-11 18:47:35


In [ ]:
# Evaluation: run PPO on test set (per-step results)

results = {}

for scenario in [SCENARIO_OVERLOAD, SCENARIO_COST]:
    comp = evaluate_scenario(
        scenario,
        episode_length=exp_config.episode_length,
    )
    results[scenario] = comp
    print_comparison(comp)

results



Evaluating scenario: OVERLOAD
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_forest
  Target: cpu_total_usage
  Saved at: 2025-11-11 22:19:30
Latest model: models\svr_memory_usage_pct_20251111_184735.pkl
✓ Model loaded: models\svr_memory_usage_pct_20251111_184735.pkl
  Model: svr
  Target: memory_usage_pct
  Saved at: 2025-11-11 18:47:35
Latest model: models\random_forest_system_load_20251111_221944.pkl
✓ Model loaded: models\random_forest_system_load_20251111_221944.pkl
  Model: random_forest
  Target: system_load
  Saved at: 2025-11-11 22:19:44
Loaded model from E:\PROJECTS\Demand-Forecasting-and-Resource-Optimization-on-Cloud-Infrastructure\rl_models\ppo_overload.zip

Running PPO rollout for scenario: overload
Latest model: models\random_forest_cpu_total_usage_20251111_221930.pkl
✓ Model loaded: models\random_forest_cpu_total_usage_20251111_221930.pkl
  Model: random_for

{'overload': {'scenario': 'overload',
  'timestamp': '2025-12-16 00:03:13',
  'ppo': {'total_vm_cost': 1288.7007999999998,
   'total_switching_cost': 118.27,
   'total_cost': 1406.9708,
   'sla_violations': 49,
   'sla_violation_rate': 0.002857142857142857,
   'mean_cpu_utilization': 0.32636006286597025,
   'mean_mem_utilization': 0.0,
   'n_steps': 17150},
  'ppo_30min': {'total_vm_cost': 0.0,
   'total_switching_cost': 0.0,
   'total_cost': 0.0,
   'sla_violations': 49,
   'sla_violation_rate': 0.0028455284552845526,
   'n_buckets': 287}},
 'cost': {'scenario': 'cost',
  'timestamp': '2025-12-16 00:05:11',
  'ppo': {'total_vm_cost': 713.7088000000001,
   'total_switching_cost': 72.06,
   'total_cost': 785.7688,
   'sla_violations': 0,
   'sla_violation_rate': 0.0,
   'mean_cpu_utilization': 0.06705591302453638,
   'mean_mem_utilization': 0.0,
   'n_steps': 17150},
  'ppo_30min': {'total_vm_cost': 0.0,
   'total_switching_cost': 0.0,
   'total_cost': 0.0,
   'sla_violations': 0,
   's

In [ ]:
# Inspect generated PPO schedules (PER-STEP results)

import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("forecast_result")

# Load PPO per-step schedules
ppo_overload_path = RESULTS_DIR / "ppo_schedule_test_overload.csv"
ppo_cost_path = RESULTS_DIR / "ppo_schedule_test_cost.csv"

print("=== PPO Per-Step Schedules ===")
print(f"PPO overload: {ppo_overload_path.exists()}")
print(f"PPO cost: {ppo_cost_path.exists()}")

ppo_overload_df = pd.read_csv(ppo_overload_path) if ppo_overload_path.exists() else None
ppo_cost_df = pd.read_csv(ppo_cost_path) if ppo_cost_path.exists() else None

if ppo_overload_df is not None:
    print(f"\nPPO Overload: {len(ppo_overload_df)} steps")
    display(ppo_overload_df.head(10))

if ppo_cost_df is not None:
    print(f"\nPPO Cost: {len(ppo_cost_df)} steps")
    display(ppo_cost_df.head(10))


=== PPO Per-Step Schedules ===
PPO overload: True
PPO cost: True

PPO Overload: 17150 steps


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0,0.0,0.0,0
5,1970-01-25 01:08:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.23,0.258050,0.0,0,0.0,0.0,0
6,1970-01-25 01:08:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.14,0.258136,0.0,0,0.0,0.0,0
7,1970-01-25 01:09:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.21,0.258079,0.0,0,0.0,0.0,0
8,1970-01-25 01:09:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.12,0.258266,0.0,0,0.0,0.0,0
9,1970-01-25 01:10:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.07,0.258260,0.0,0,0.0,0.0,0



PPO Cost: 17150 steps


,timestamp,allocation,vm_cost_per_hour,switching_cost,total_cost_per_hour,cpu_allocated_cores,mem_allocated_gb,cpu_vm_only,mem_vm_only,cpu_required_cores,mem_required_gb,cpu_overflow_cores,mem_overflow_gb,cpu_utilization_pct,mem_utilization_pct,sla_violation_flag
0,1970-01-25 01:05:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.24,0.258175,0.0,0,0.0,0.0,0
1,1970-01-25 01:06:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.22,0.258186,0.0,0,0.0,0.0,0
2,1970-01-25 01:06:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.13,0.258044,0.0,0,0.0,0.0,0
3,1970-01-25 01:07:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.08,0.257840,0.0,0,0.0,0.0,0
4,1970-01-25 01:07:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.10,0.258327,0.0,0,0.0,0.0,0
5,1970-01-25 01:08:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.23,0.258050,0.0,0,0.0,0.0,0
6,1970-01-25 01:08:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.14,0.258136,0.0,0,0.0,0.0,0
7,1970-01-25 01:09:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.21,0.258079,0.0,0,0.0,0.0,0
8,1970-01-25 01:09:30,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.12,0.258266,0.0,0,0.0,0.0,0
9,1970-01-25 01:10:00,Host only,0.0,0.0,0.0,0.7,3.0,0.0,0.0,0.07,0.258260,0.0,0,0.0,0.0,0
